In [1]:
print(123)

123


In [2]:
import sys
sys.path.insert(0, '..')
from agentic_rag.ingest import load_faq_data

documents = load_faq_data()

In [3]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

85

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
from google import genai
from google.genai import types
google_ai_client = genai.Client(api_key=os.getenv("GOOGLE_AI_API_KEY_CLEAN"))

In [18]:
import json

user_prompt = json.dumps(doc)

In [19]:
contents = [
    {"role": "user", "parts": [{"text": user_prompt}]}
]

In [27]:
response = google_ai_client.models.generate_content(
            model='gemini-2.5-flash',
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=data_gen_instructions,
                response_mime_type="application/json",
                response_schema=Questions
            ),        
)


In [30]:
response.parsed.questions

['Is it still possible to enroll in the course now?',
 'What are the requirements to get a certificate after joining the course late?',
 'Do I need to complete a project to earn a certificate?',
 'Is there a specific timeframe for project submissions to qualify for a certificate?',
 'Can someone who just discovered the course still obtain a certificate?']

In [31]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [32]:
from evaluation_utils import llm_structured

In [33]:
result, usage = llm_structured(
    google_ai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

["Is it still possible to join the course if I'm starting late?", 'What are the requirements for getting a certificate?', 'Do I need to submit a project to earn the certificate?', 'Is there a deadline for submitting the project?', 'Can late enrollees still qualify for a course certificate?']


In [35]:
usage.prompt_token_count, usage.candidates_token_count

(174, 65)

In [39]:
from evaluation_utils import calc_price

In [40]:
cost = calc_price(usage)

cost

{'input_cost': 1.305e-05,
 'output_cost': 1.9499999999999996e-05,
 'total_cost': 3.255e-05}

In [41]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': "Is it still possible to join the course if I'm starting late?",
  'document': '74eb249bbf'},
 {'question': 'What are the requirements for getting a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit a project to earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for submitting the project?',
  'document': '74eb249bbf'},
 {'question': 'Can late enrollees still qualify for a course certificate?',
  'document': '74eb249bbf'}]

In [43]:
import pandas as pd

In [44]:
pd.DataFrame(records)

,question,document
0,Is it still possible to join the course if I'm...,74eb249bbf
1,What are the requirements for getting a certif...,74eb249bbf
2,Do I need to submit a project to earn the cert...,74eb249bbf
3,Is there a deadline for submitting the project?,74eb249bbf
4,Can late enrollees still qualify for a course ...,74eb249bbf


In [45]:
from evaluation_utils import llm_structured_retry

In [58]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        google_ai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model = "gemini-2-flash-lite",
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [52]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [55]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [59]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/85 [00:00<?, ?it/s]

KeyboardInterrupt: 